## 1. Understanding Data

In [ ]:
import os
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings(action='ignore')

### Before Start
????? ???? ?? ???? ??? ?? ??? ?? ?? ????? ??? ????? ???.

### Q. ???? CSV??? ??? ??? ?? ??? ???? ??? ?? ? ?? ???? ????
??? ??? ?? ?? ??? ???? ?? ??? ? ????.   
??? CSV ???? 1GB ?? ????? ???? read? ?? 2~3GB? ???? ??? ?? ???, ??? ??????

### Load data

In [ ]:
PATH = '../input/kakr-4th-competition/'
train = pd.read_csv(PATH + 'train.csv')
test  = pd.read_csv(PATH + 'test.csv')

### ??? ??

* id
* age : ??
* workclass : ?? ??
* fnlwgt : ?? ???? ???? ??? (final weight? ??)
* education : ?? ??
* education_num : ?? ?? ??
* marital_status: ?? ??
* occupation : ??
* relationship : ?? ??
* race : ??
* sex : ??
* capital_gain : ?? ??
* capital_loss : ?? ??
* hours_per_week : ?? ?? ??
* native_country : ??
* income : ?? (???? ?? ?)
    * \>50K : 1
    * <=50K : 0

### Pandas ?????
Pandas ??????? ?? ??? ??? ??? ????? ?????.

loc, iloc, info, describe, value_counts, head, tail, sample, map, apply, groupby ?

In [ ]:
train.shape

In [ ]:
train.apply(lambda x: x['age'] + x['hours_per_week'], axis=1)

In [ ]:
## ??? ?? ?
train.describe()

In [ ]:
train.capital_gain < 99999

In [ ]:
train

In [ ]:
train.income = train.income.map(lambda x: int(x=='>50K'))

In [ ]:
train.age.plot(kind='box')

## 2. Data Preprocessing

### 2.1 ??? ??

In [ ]:
train[train.apply(lambda x: "?" in list(x), axis=1)]

In [ ]:
train.occupation.value_counts()

In [ ]:
train.workclass.value_counts()

### ??? ??
??? ???

In [ ]:
train[(train['workclass'] == '?')&(train['occupation'] == '?')]

In [ ]:
train.loc[(train.age >= 30)&(train.workclass=='?'), 'workclass'] = 'No'

In [ ]:
train.workclass.value_counts()

In [ ]:
train.native_country.value_counts()

### Null check ?? ?
???? ??? ? ???? ?? ???? ??? ???? ????

In [ ]:
train.info()

### 2.2 ??? ??
??? ???? ??? ?? ??? ?? ?????? ?????.

In [ ]:
train.describe()

In [ ]:
train.loc[train.capital_gain < 99999].sort_values('capital_gain', ascending=False)

### log ??

In [ ]:
train["log_capital_gain"] = train.capital_gain.map(lambda x: np.log(x, where=(x!=0)))

In [ ]:
train[['capital_gain', 'log_capital_gain']].describe()

In [ ]:
train.capital_gain.map(lambda x: np.log(x, where=(x!=0))).describe()

### 2.3 Scaling

Min-max Scaler: 
??? ??? ?? ??? ?  
?????? ???
  
Standard Scaler: 
??? 0, ????? 1? ??? ????? ??? ???? ??  
?????? ??? ? ??

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

mm_scaler = MinMaxScaler()
st_scaler = StandardScaler()

In [ ]:
train['MM_fnlwgt'] = mm_scaler.fit_transform(train['fnlwgt'].values.reshape(-1,1))
test['MM_fnlwgt'] = mm_scaler.transform(test['fnlwgt'].values.reshape(-1,1))

train['MM_age'] = mm_scaler.fit_transform(train['age'].values.reshape(-1,1))
test['MM_age'] = mm_scaler.transform(test['age'].values.reshape(-1,1))

train['ST_fnlwgt'] = st_scaler.fit_transform(train['fnlwgt'].values.reshape(-1,1))
test['ST_fnlwgt'] = st_scaler.transform(test['fnlwgt'].values.reshape(-1,1))

train['ST_age'] = st_scaler.fit_transform(train['age'].values.reshape(-1,1))
test['ST_age'] = st_scaler.transform(test['age'].values.reshape(-1,1))

In [ ]:
train.describe()[['MM_fnlwgt', 'MM_age']]

In [ ]:
train.describe()[['ST_fnlwgt', 'ST_age']].round(6)

## 3. Feature Engineering

### 3.1 ??? ??
?? ??? ??? ?? ???? ??? ?????.

In [ ]:
train

### 3.2 ???
????? ???? ??? Feature?? ??? ??? ? ?????

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [ ]:
train

### Onehot encoder

In [ ]:
oe = OneHotEncoder()
oe_result = oe.fit_transform(train['workclass'].values.reshape(-1, 1))

In [ ]:
oe.get_feature_names(['workclass'])

In [ ]:
oe_result

In [ ]:
sub = pd.DataFrame(data=oe_result.toarray(), columns=oe.get_feature_names(['workclass']))

In [ ]:
pd.concat([train, sub], axis=1)

### Label encoder

In [ ]:
le = LabelEncoder()
le.fit_transform(train['workclass'].values.reshape(-1, 1))

In [ ]:
workclass_to_num = dict(zip(train['workclass'].unique(), [0,1,2,3,4,5,6,7,8,9]))

In [ ]:
train['workclass'].map(workclass_to_num)

In [ ]:
train['workclass'].unique()

???? ???? ????? ?? ???.. ?? ??? ? ?? ?? ????.

### (Target) Mean Encoding

In [ ]:
train

In [ ]:
male_positive = train.loc[train['sex']=='Male', "income"]
female_positive = train.loc[train['sex']=='Female', "income"]

In [ ]:
(male_positive.value_counts() / male_positive.shape[0]).iloc[1]

In [ ]:
female_positive.value_counts() / female_positive.shape[0]

### onehot ??? ????? ?? ??? ??~

In [ ]:
dummied = pd.get_dummies(train)

In [ ]:
dummied

### 3.3 PCA

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=60, svd_solver='full')

In [ ]:
dummied = dummied.drop(columns=['id', 'income_<=50K', 'income_>50K'])

In [ ]:
X_train_std = st_scaler.fit_transform(dummied)

In [ ]:
X_train_std = st_scaler.fit_transform(dummied)
X_train_pca = pca.fit_transform(X_train_std)

In [ ]:
X_train_pca.shape

In [ ]:
X_train_2.shape

In [ ]:
pca.explained_variance_ratio_

In [ ]:
X_train_2 = X_train_std - X_train_std.mean(axis=0)

res = np.dot(X_train_2, pca.components_.T)
res

In [ ]:
np.allclose(X_train_pca, res)

## 4. Before Modeling
??? ????..